In [ ]:
#imports
import numpy as np
import gymnasium as gym
from S7.episodes import episode_deterministe, episode_stochastic



#Définition d'environnement 
env = gym.make("FrozenLake-v1", is_slippery=False)


nS = env.observation_space.n  # type: ignore
nA = env.action_space.n  # type: ignore

#Actions
UP = 0
RIGHT = 1
DOWN = 2
LEFT = 3


### Monte Carlo First Visit Prediciton

In [65]:
def mc_first_visit_prediction(env,policy,n_episodes=200, gamma=0.9):
    V= np.zeros(nS)
    N=np.zeros(nS)
    
    for _ in range(n_episodes):
       episode =  episode_deterministe(env, policy)
       G = 0.0
       visited = set()
       for t in range(len(episode) - 1, -1, -1):
           s,a,r = episode[t]
           G = r + gamma * G
           if s not in visited :
               visited.add(s)
               N[s]+=1
               V[s]+=(1/N[s])* (G - V[s])
    return V

Exemple

In [66]:
# Define a simple random policy
def random_policy(state):
    """Returns a random action for any state"""
    return np.random.randint(0, nA)

# Define a deterministic policy (e.g., always go right then down)
def example_policy(state):
    """A simple deterministic policy"""
    # This policy tries to move right or down based on state
    if state < 4:
        return RIGHT  # Go right in first row
    else:
        return DOWN   # Go down in other rows

# Apply the Monte Carlo First-Visit Prediction
V_random = mc_first_visit_prediction(env, random_policy, n_episodes=500, gamma=0.99)
V_example = mc_first_visit_prediction(env, example_policy, n_episodes=500, gamma=0.99)

# Display results
print("State Value Function (Random Policy):")
print(V_random.reshape(4, 4))
print("\nState Value Function (Example Policy):")
print(V_example.reshape(4, 4))

# Value of starting state (typically state 0)
print(f"\nValue of starting state (Random Policy): {V_random[0]:.4f}")
print(f"Value of starting state (Example Policy): {V_example[0]:.4f}")

State Value Function (Random Policy):
[[0.01118685 0.00318078 0.00724104 0.        ]
 [0.01483664 0.         0.026136   0.        ]
 [0.03264183 0.08445343 0.15631579 0.        ]
 [0.         0.11947413 0.46153846 0.        ]]

State Value Function (Example Policy):
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]

Value of starting state (Random Policy): 0.0112
Value of starting state (Example Policy): 0.0000


### Monte Carlo on-policy Control

In [67]:
def mc_control_on_policy(env,epsilon=0.1, n_episodes=200, gamma=0.9):
    Q=np.zeros((nS,nA))
    N = np.zeros((nS,nA))
    pi=np.ones((nS,nA)) / nA  # Initial policy: uniform random
    for _ in range (n_episodes):
        episode = episode_stochastic(env, pi)
        G=0.0 #Retour G_T
        visited = set()
        for t in range(len(episode) - 1, -1, -1):
            s,a,r = episode[t]
            G = r+gamma*G
            if (s,a) not in visited :
                visited.add((s,a))
                N [s,a] +=1
                Q[s,a] +=(1/N[s,a]*(G -Q[s,a]))
                pi[s] = np.ones(nA)* (epsilon/nA)
                pi[s,np.argmax(Q[s])]+= (1-epsilon)
    policy = np.argmax(Q, axis=1)     
    return policy

Exemple

In [68]:
# Train the Monte Carlo on-policy control algorithm
learned_policy = mc_control_on_policy(env, epsilon=0.1, n_episodes=1000, gamma=0.99)

# Display the learned policy
print("Learned Policy (best action for each state):")
print("Action mapping: UP=0, RIGHT=1, DOWN=2, LEFT=3")
print(learned_policy)
print("\nLearned Policy Grid (4x4):")
action_names = {0: "UP", 1: "RIGHT", 2: "DOWN", 3: "LEFT"}
policy_grid = np.array([[action_names[learned_policy[i*4 + j]] for j in range(4)] for i in range(4)])
print(policy_grid)

# Test the learned policy by running a few episodes
def evaluate_policy(env, policy, n_test_episodes=10):
    """Test a policy and return average reward and success rate"""
    total_reward = 0
    successes = 0
    
    for _ in range(n_test_episodes):
        state, _ = env.reset()
        done = False
        episode_reward = 0
        
        while not done:
            action = policy[state]
            state, reward, terminated, truncated, _ = env.step(action)
            episode_reward += reward
            done = terminated or truncated
        
        total_reward += episode_reward
        if episode_reward > 0:  # Reached the goal
            successes += 1
    
    return total_reward / n_test_episodes, successes / n_test_episodes * 100

avg_reward, success_rate = evaluate_policy(env, learned_policy, n_test_episodes=50)
print(f"\n--- Evaluation Results ---")
print(f"Average Reward: {avg_reward:.4f}")
print(f"Success Rate: {success_rate:.2f}%")

Learned Policy (best action for each state):
Action mapping: UP=0, RIGHT=1, DOWN=2, LEFT=3
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]

Learned Policy Grid (4x4):
[['UP' 'UP' 'UP' 'UP']
 ['UP' 'UP' 'UP' 'UP']
 ['UP' 'UP' 'UP' 'UP']
 ['UP' 'UP' 'UP' 'UP']]

--- Evaluation Results ---
Average Reward: 0.0000
Success Rate: 0.00%


### Monte Carlo off-policy Control

In [69]:
def mc_off_policy_control(env, b, n_episodes, gamma=0.9):
    Q = np.zeros((nS, nA))
    C = np.zeros((nS, nA))

    # comportement policy sous forme matricielle
    if callable(b):
        pi_b = np.array([b(s) for s in range(nS)])
    else:
        pi_b = b

    for _ in range(n_episodes):
        episode = episode_stochastic(env, pi_b)
        G = 0.0
        W = 1.0

        for t in range(len(episode) - 1, -1, -1):
            s, a, r = episode[t]
            G = r + gamma * G
            C[s, a] += W
            Q[s, a] += (W / C[s, a]) * (G - Q[s, a])

            greedy_a = np.argmax(Q[s])

            if a != greedy_a:
                break

            W *= 1.0 / pi_b[s][a]

    policy = np.argmax(Q, axis=1)
    return policy

Exemple

In [ ]:
# Define a behavior policy (soft policy for exploration)
def behavior_policy(state):
    epsilon = 0.5
    probs = np.ones(nA) / nA # la somme doit être égale à 1 pour que ce soit une distribution de probabilité 
    return probs


# Train Monte Carlo off-policy control with importance sampling
learned_policy_offpolicy = mc_off_policy_control(env, behavior_policy, n_episodes=1000, gamma=0.99)

# Display the learned policy
print("Learned Off-Policy Control Policy:")
print("Action mapping: UP=0, RIGHT=1, DOWN=2, LEFT=3")
print(learned_policy_offpolicy)
print("\nLearned Off-Policy Policy Grid (4x4):")
policy_grid_offpolicy = np.array([[action_names[learned_policy_offpolicy[i*4 + j]] for j in range(4)] for i in range(4)])
print(policy_grid_offpolicy)

# Evaluate the off-policy learned policy
avg_reward_offpolicy, success_rate_offpolicy = evaluate_policy(env, learned_policy_offpolicy, n_test_episodes=50)
print(f"\n--- Off-Policy Evaluation Results ---")
print(f"Average Reward: {avg_reward_offpolicy:.4f}")
print(f"Success Rate: {success_rate_offpolicy:.2f}%")

# Comparison
print(f"\n--- Comparison ---")
print(f"On-Policy Success Rate: {success_rate:.2f}%")
print(f"Off-Policy Success Rate: {success_rate_offpolicy:.2f}%")

Learned Off-Policy Control Policy:
Action mapping: UP=0, RIGHT=1, DOWN=2, LEFT=3
[1 2 1 0 1 0 1 0 2 2 1 0 0 2 2 0]

Learned Off-Policy Policy Grid (4x4):
[['RIGHT' 'DOWN' 'RIGHT' 'UP']
 ['RIGHT' 'UP' 'RIGHT' 'UP']
 ['DOWN' 'DOWN' 'RIGHT' 'UP']
 ['UP' 'DOWN' 'DOWN' 'UP']]

--- Off-Policy Evaluation Results ---
Average Reward: 1.0000
Success Rate: 100.00%

--- Comparison ---
On-Policy Success Rate: 0.00%
Off-Policy Success Rate: 100.00%


### TD0 prediction

In [72]:
def td_zero_prediction(env,policy, n_episodes=200, gamma=0.9, alpha=0.1):
    V = np.zeros(nS)
    for _ in range(n_episodes):
        state, _ = env.reset()
        done = False
        while not done:
            action = policy(state) if callable(policy) else int(policy[state])
            next_state, reward, terminated, truncated, _ = env.step(action)
            V_next= 0.0 if terminated else V[next_state]
            delta = reward + gamma * V_next - V[state]
            V[state] += alpha * delta
            state = next_state
            done = terminated or truncated
           
    return V

In [73]:
# Apply TD(0) Prediction with different policies
V_td_random = td_zero_prediction(env, random_policy, n_episodes=500, gamma=0.99, alpha=0.1)
V_td_example = td_zero_prediction(env, example_policy, n_episodes=500, gamma=0.99, alpha=0.1)

# Display results
print("=== TD(0) Prediction Results ===\n")
print("State Value Function (Random Policy):")
print(V_td_random.reshape(4, 4))
print("\nState Value Function (Example Policy):")
print(V_td_example.reshape(4, 4))

# Value of starting state
print(f"\nValue of starting state (Random Policy): {V_td_random[0]:.4f}")
print(f"Value of starting state (Example Policy): {V_td_example[0]:.4f}")

# Comparison with Monte Carlo
print("\n=== Comparison: Monte Carlo vs TD(0) ===")
print(f"MC (Random Policy): {V_random[0]:.4f} vs TD(0): {V_td_random[0]:.4f}")
print(f"MC (Example Policy): {V_example[0]:.4f} vs TD(0): {V_td_example[0]:.4f}")

print("\nKey differences:")
print("- Monte Carlo: Learns from complete episodes (offline)")
print("- TD(0): Updates after each step using bootstrapping (online)")
print("- TD(0) typically converges faster with fewer samples")

=== TD(0) Prediction Results ===

State Value Function (Random Policy):
[[0.00183264 0.00081906 0.00096096 0.00055406]
 [0.00215944 0.         0.00140633 0.        ]
 [0.00522478 0.01929271 0.028964   0.        ]
 [0.         0.06941947 0.23319317 0.        ]]

State Value Function (Example Policy):
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]

Value of starting state (Random Policy): 0.0018
Value of starting state (Example Policy): 0.0000

=== Comparison: Monte Carlo vs TD(0) ===
MC (Random Policy): 0.0112 vs TD(0): 0.0018
MC (Example Policy): 0.0000 vs TD(0): 0.0000

Key differences:
- Monte Carlo: Learns from complete episodes (offline)
- TD(0): Updates after each step using bootstrapping (online)
- TD(0) typically converges faster with fewer samples


### Q-learning

In [74]:
def q_learning(env, n_episodes=200, gamma=0.9, alpha=0.1, epsilon=0.1):
    Q = np.zeros((nS, nA))
    for _ in range(n_episodes):
        s, _ = env.reset()
        done = False
        while not done:
            probs = np.ones(nA) * (epsilon / nA)  # Epsilon-greedy exploration
            probs[np.argmax(Q[s])] += (1 - epsilon)
            a = np.random.choice(np.arange(nA), p=probs)
            s_next, r, terminated, truncated, _ = env.step(a)
            Q_max = 0.0 if terminated else np.max(Q[s_next])
            Q[s, a] += alpha * (r + gamma * Q_max - Q[s, a])
            s = s_next
            done = terminated or truncated
            
    policy = np.argmax(Q, axis=1)
    return policy

Exemple

In [75]:
# Train Q-learning algorithm
learned_policy_qlearning = q_learning(env, n_episodes=1000, gamma=0.99, alpha=0.1, epsilon=0.1)

# Display the learned policy
print("=== Q-Learning Results ===\n")
print("Learned Q-Learning Policy:")
print("Action mapping: UP=0, RIGHT=1, DOWN=2, LEFT=3")
print(learned_policy_qlearning)
print("\nLearned Q-Learning Policy Grid (4x4):")
policy_grid_qlearning = np.array([[action_names[learned_policy_qlearning[i*4 + j]] for j in range(4)] for i in range(4)])
print(policy_grid_qlearning)

# Evaluate the Q-learning policy
avg_reward_qlearning, success_rate_qlearning = evaluate_policy(env, learned_policy_qlearning, n_test_episodes=50)
print(f"\n--- Q-Learning Evaluation Results ---")
print(f"Average Reward: {avg_reward_qlearning:.4f}")
print(f"Success Rate: {success_rate_qlearning:.2f}%")

# Comprehensive comparison of all algorithms
print(f"\n=== Algorithm Comparison ===")
print(f"{'Algorithm':<25} {'Success Rate':<20}")
print("-" * 45)
print(f"{'MC On-Policy':<25} {success_rate:.2f}%")
print(f"{'MC Off-Policy':<25} {success_rate_offpolicy:.2f}%")
print(f"{'Q-Learning':<25} {success_rate_qlearning:.2f}%")

print("\nKey Points:")
print("- Q-Learning: Off-policy, model-free temporal difference learning")
print("- Updates Q-values using the max next Q-value (greedy)")
print("- Can be more sample-efficient than Monte Carlo")
print("- Uses epsilon-greedy for exploration during training")

=== Q-Learning Results ===

Learned Q-Learning Policy:
Action mapping: UP=0, RIGHT=1, DOWN=2, LEFT=3
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]

Learned Q-Learning Policy Grid (4x4):
[['UP' 'UP' 'UP' 'UP']
 ['UP' 'UP' 'UP' 'UP']
 ['UP' 'UP' 'UP' 'UP']
 ['UP' 'UP' 'UP' 'UP']]

--- Q-Learning Evaluation Results ---
Average Reward: 0.0000
Success Rate: 0.00%

=== Algorithm Comparison ===
Algorithm                 Success Rate        
---------------------------------------------
MC On-Policy              0.00%
MC Off-Policy             100.00%
Q-Learning                0.00%

Key Points:
- Q-Learning: Off-policy, model-free temporal difference learning
- Updates Q-values using the max next Q-value (greedy)
- Can be more sample-efficient than Monte Carlo
- Uses epsilon-greedy for exploration during training


### SARSA (Prediction)

In [76]:
def sarsa(env, nS, nA, n_episodes=200, gamma=0.9, alpha=0.1, epsilon=0.1):
    Q = np.zeros((nS, nA))

    for _ in range(n_episodes):
        s, _ = env.reset()

        # choose first action using epsilon-greedy from Q
        probs = np.ones(nA) * (epsilon / nA)
        probs[np.argmax(Q[s])] += (1 - epsilon)
        a = np.random.choice(np.arange(nA), p=probs)

        done = False
        while not done:
            s_next, r, terminated, truncated, _ = env.step(a)
            done = terminated or truncated

            if done:
                target = r
            else:
                # choose next action using epsilon-greedy from Q
                probs = np.ones(nA) * (epsilon / nA)
                probs[np.argmax(Q[s_next])] += (1 - epsilon)
                a_next = np.random.choice(np.arange(nA), p=probs)

                target = r + gamma * Q[s_next, a_next]

            Q[s, a] += alpha * (target - Q[s, a])

            if not done:
                s, a = s_next, a_next

    policy = np.argmax(Q, axis=1)
    return policy

In [77]:
# Train SARSA algorithm
learned_policy_sarsa = sarsa(env, nS, nA, n_episodes=1000, gamma=0.99, alpha=0.1, epsilon=0.1)

# Display the learned policy
print("=== SARSA Results ===\n")
print("Learned SARSA Policy:")
print("Action mapping: UP=0, RIGHT=1, DOWN=2, LEFT=3")
print(learned_policy_sarsa)
print("\nLearned SARSA Policy Grid (4x4):")
policy_grid_sarsa = np.array([[action_names[learned_policy_sarsa[i*4 + j]] for j in range(4)] for i in range(4)])
print(policy_grid_sarsa)

# Evaluate the SARSA policy
avg_reward_sarsa, success_rate_sarsa = evaluate_policy(env, learned_policy_sarsa, n_test_episodes=50)
print(f"\n--- SARSA Evaluation Results ---")
print(f"Average Reward: {avg_reward_sarsa:.4f}")
print(f"Success Rate: {success_rate_sarsa:.2f}%")

# Comprehensive comparison of all TD algorithms
print(f"\n=== TD-based Algorithms Comparison ===")
print(f"{'Algorithm':<25} {'Success Rate':<20}")
print("-" * 45)
print(f"{'TD(0) Prediction':<25} {success_rate:.2f}%")
print(f"{'Q-Learning (Off-Policy)':<25} {success_rate_qlearning:.2f}%")
print(f"{'SARSA (On-Policy)':<25} {success_rate_sarsa:.2f}%")

print("\nKey Differences:")
print("- Q-Learning: Off-policy, uses max(Q[s_next]) - greedy")
print("- SARSA: On-policy, uses actual next action Q[s_next, a_next]")
print("- SARSA tends to be more conservative (follows exploration policy)")
print("- Q-Learning can learn from suboptimal trajectories more effectively")

=== SARSA Results ===

Learned SARSA Policy:
Action mapping: UP=0, RIGHT=1, DOWN=2, LEFT=3
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]

Learned SARSA Policy Grid (4x4):
[['UP' 'UP' 'UP' 'UP']
 ['UP' 'UP' 'UP' 'UP']
 ['UP' 'UP' 'UP' 'UP']
 ['UP' 'UP' 'UP' 'UP']]

--- SARSA Evaluation Results ---
Average Reward: 0.0000
Success Rate: 0.00%

=== TD-based Algorithms Comparison ===
Algorithm                 Success Rate        
---------------------------------------------
TD(0) Prediction          0.00%
Q-Learning (Off-Policy)   0.00%
SARSA (On-Policy)         0.00%

Key Differences:
- Q-Learning: Off-policy, uses max(Q[s_next]) - greedy
- SARSA: On-policy, uses actual next action Q[s_next, a_next]
- SARSA tends to be more conservative (follows exploration policy)
- Q-Learning can learn from suboptimal trajectories more effectively
